# Initialize External Fusion Codes for VAFT

VAFT reads runtime settings directly from the process environment. Configure them after installation in a shell startup file, a Conda activation script, an environment module, or the equivalent mechanism used by your computing site.

VAFT uses two naming conventions:

- External scientific-code installations use `{CODE}HOME` and point to the repository or installation root.
- Paths and options owned by VAFT use descriptive `VAFT_*` names.

VAFT does not provide a separate runtime-configuration registry. External-code licensing, installation, and compilation remain the responsibility of the user and the upstream project.

## External scientific-code roots

When a home variable is defined, its adapter uses the exact executable paths below. A missing or non-executable file is an installation error; VAFT does not silently try another variable or `PATH`.

| Variable | Installation root | Expected executables | Consuming adapter |
| --- | --- | --- | --- |
| `GPECHOME` | GPEC root | `bin/dcon`, `bin/match`, `bin/rdcon`, `bin/stride`, `bin/gpec` | `vaft.code.gpec` |
| `CHEASEHOME` | CHEASE root | `bin/chease` | `vaft.code.chease` |
| `EFITHOME` | EFIT root | `bin/efit` | `vaft.code.efit` |
| `TESHOME` | TES root | `bin/rtes` | `vaft.code.tes` |

Expected layout:

```text
$GPECHOME/bin/{dcon,match,rdcon,stride,gpec}
$CHEASEHOME/bin/chease
$EFITHOME/bin/efit
$TESHOME/bin/rtes
```

Compile or install each project so these files exist and have execute permission.

## Persistent initialization

Set only the roots for codes you use. For Bash or Zsh, add entries such as these to `~/.bashrc` or `~/.zshrc`, then start a new shell or source the file:

```bash
export GPECHOME=/opt/gpec
export CHEASEHOME=/opt/chease
export EFITHOME=/opt/efit
export TESHOME=/opt/tes
export VAFT_FILEDB_DIR=/data/VEST/FileDB
```

For a Conda environment, put the same exports in `$CONDA_PREFIX/etc/conda/activate.d/vaft.sh`. If the environment is not active while creating the file, replace `$CONDA_PREFIX` with its absolute path.

On Windows, set them as *user* environment variables. The registered "Python (vaft)" kernel starts the environment's `python.exe` directly rather than through Conda activation, so an `activate.d` script never reaches a notebook, while a user variable reaches every newly started process:

```powershell
[Environment]::SetEnvironmentVariable('GPECHOME', "$env:LOCALAPPDATA\vaft\external\gpec", 'User')
[Environment]::SetEnvironmentVariable('CHEASEHOME', "$env:LOCALAPPDATA\vaft\external\chease", 'User')
$env:GPECHOME = "$env:LOCALAPPDATA\vaft\external\gpec"
```

Open a new terminal, or restart JupyterLab, for it to take effect elsewhere. The executables under `bin/` are named `dcon.exe`, `chease.exe` and so on there; VAFT resolves the documented POSIX name to the native build beside it, so the layouts below are written the same way on every platform. `install/README.md` covers building the codes natively on Windows.

A site environment-module file can use:

```tcl
#%Module1.0
setenv GPECHOME /opt/gpec
setenv CHEASEHOME /opt/chease
setenv EFITHOME /opt/efit
setenv TESHOME /opt/tes
setenv VAFT_FILEDB_DIR /data/VEST/FileDB
```

Setting `os.environ` inside a notebook affects only that Python process and is therefore not the recommended persistent setup.

## Verify the active environment

Run the following read-only check after initializing your shell. Unconfigured codes are reported as optional; configured roots are checked for every executable expected by VAFT.

In [ ]:
import os
from pathlib import Path

# `os.access(path, os.X_OK)` is true for every readable file on Windows, so it
# cannot tell a real program from a text file there, and it does not know that
# the documented `bin/dcon` is `bin\dcon.exe` on a native build. These two
# helpers answer both questions the way the running platform does.
from vaft.compat import is_executable, resolve_executable

external_codes = {
    # rmatch is the companion RDCON runs, exactly as match follows DCON.
    "GPECHOME": ("bin/dcon", "bin/match", "bin/rdcon", "bin/rmatch", "bin/stride", "bin/gpec"),
    "CHEASEHOME": ("bin/chease",),
    "EFITHOME": ("bin/efit",),
    "TESHOME": ("bin/rtes",),
}

for variable, relative_executables in external_codes.items():
    configured = os.environ.get(variable)
    if not configured:
        print(f"{variable}: not configured")
        continue

    root = Path(configured).expanduser()
    problems = []
    for relative in relative_executables:
        expected = root / relative
        executable = resolve_executable(expected)
        if executable is None:
            problems.append(f"missing {expected}")
        elif not is_executable(executable):
            problems.append(f"not executable: {executable}")

    status = "ready" if not problems else "; ".join(problems)
    print(f"{variable}={root}: {status}")

GPECHOME=~/git/GPEC: ready
CHEASEHOME: not configured
EFITHOME: not configured
TESHOME: not configured


## Backward-compatible executable variables

Existing installations may retain the following variables when the corresponding `{CODE}HOME` variable is unset. New installations should use `{CODE}HOME`.

| Variable | Value | Behavior |
| --- | --- | --- |
| `EFIT` | EFIT executable or directory containing `efit` | Used only when `EFITHOME` is unset. |
| `CHEASE` | CHEASE executable or directory containing `chease` | Used only when `CHEASEHOME` is unset. |
| `CHEASE_EXEC_DIR` | Directory containing `chease`, or an executable path accepted by the adapter | Used after `CHEASE` when `CHEASEHOME` is unset. |
| `RTES` | Path to `rtes` | Used only when `TESHOME` is unset. |

An explicitly supplied executable in an existing adapter configuration also remains supported. These are compatibility paths, not a global configuration-precedence system.

## Python-package codes: TokaMaker (Open FUSION Toolkit)

TokaMaker is driven in-process through the `OpenFUSIONToolkit` Python package
(`vaft.code.tokamaker`), so there is no `{CODE}HOME` executable root. Install the
package from a compiled Open FUSION Toolkit checkout or release:

```bash
pip install -e /path/to/OpenFUSIONToolkit/src/python
```

Three optional environment variables steer discovery when the plain import fails:

| Variable | Purpose |
| --- | --- |
| `OFT_LIBRARY_DIR` | Directory containing the compiled `liboftpy` library |
| `OFT_INSTALL_DIR` | Toolkit install root containing `bin/liboftpy` |
| `OFT_ROOTPATH` | OFT release/source root; its `python` (or `src/python`) directory is added to `sys.path` automatically |


In [ ]:
import importlib.util

spec = importlib.util.find_spec("OpenFUSIONToolkit")
if spec is None:
    print("OpenFUSIONToolkit: NOT importable (TokaMaker workflows unavailable)")
else:
    print(f"OpenFUSIONToolkit: importable from {spec.origin}")


## VAFT-owned settings and data inputs

| Variable | Value type | Required | Purpose |
| --- | --- | --- | --- |
| `VAFT_FILEDB_DIR` | Directory path | For local FileDB workflows | Root of the canonical OMAS-first FileDB. |

OPEN-ADAS cache paths and archived raw-data sources are call inputs, not process-wide settings:

- Pass `cache_dir=` to OPEN-ADAS helpers to override their platform user-cache default.
- Pass `raw_source=` to diagnostic machine-mapping helpers, or `sample_opt=` to the low-level raw database helper, to read an archived raw dump. A path template may contain `{shot}`.
- When an archived source is supplied, VAFT does not fall back to live SQL.

## Machine-specific and dependency-owned environment

The VEST soft-X-ray integration retains `VEST_SXR_GEOMETRY_TABLE` for an explicit geometry-table file and `VEST_SXR_GEOMETRY_DIR` for a directory searched for geometry data.

VAFT also respects settings owned by dependencies or the operating system; it does not rename them:

| Variable | Owner and purpose |
| --- | --- |
| `IMAS_DD_VERSION_CONVERSION` | IMAS/OMAS data-dictionary version for conversion. |
| `IMAS_DD_CONVERSION` | Legacy fallback for `IMAS_DD_VERSION_CONVERSION`. |
| `IMAS_VERSION` | Default IMAS version in compatibility APIs. |
| `OMAS_DEBUG_TOPIC` | OMAS debug topics such as `imas_code`. |
| `OMP_NUM_THREADS` | OpenMP thread count; the EFIT adapter defaults it to `1` when unset. |
| `USER`, `HOME` | Operating-system identity and home defaults. |
| `XDG_CACHE_HOME` | Linux cache root used by OPEN-ADAS. |
| `LOCALAPPDATA` | Windows cache root used by OPEN-ADAS. |

HSDS credentials use the dependency-owned `hsconfigure` and `.hscfg` mechanism described in the main README; they are not VAFT environment variables.

In [ ]:
from pathlib import Path
import os

from vaft.compat import is_executable, resolve_executable

# Resolve against the repository root, not the kernel's cwd: Jupyter starts
# in notebooks/, which would otherwise create a nested notebooks/notebooks/.
_repo_root = Path.cwd()
if _repo_root.name == "notebooks":
    _repo_root = _repo_root.parent
OUTPUT_DIR = Path(os.environ.get(
    "VAFT_DOCS_OUTPUT_DIR", _repo_root / "notebooks" / "outputs" / "docs"
))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Select the rendering backend explicitly. In a Jupyter kernel (Jupyter Lab,
# VS Code) ask for the inline backend rather than relying on ipykernel to
# preset MPLBACKEND, which older releases do not do. With no IPython at all
# (a plain `python` run) fall back to headless Agg -- export MPLBACKEND,
# e.g. MPLBACKEND=macosx, to get windows from a script instead.
try:
    _ipython = get_ipython()
except NameError:
    _ipython = None
if _ipython is None:
    os.environ.setdefault("MPLBACKEND", "Agg")
elif "IPKernelApp" in _ipython.config:
    _ipython.run_line_magic("matplotlib", "inline")

import json
report = {}
for variable, relative_executables in external_codes.items():
    configured = os.environ.get(variable)
    if not configured:
        report[variable] = {"status": "not configured", "required": False}
        continue
    root = Path(configured).expanduser()
    # Same probe as the readiness cell above: the documented POSIX name may
    # resolve to a native .exe, and Windows cannot answer os.access().
    missing = []
    for relative in relative_executables:
        executable = resolve_executable(root / relative)
        if executable is None or not is_executable(executable):
            missing.append(str(root / relative))
    report[variable] = {
        "status": "ready" if not missing else "incomplete",
        "missing_or_not_executable": missing,
    }
text = json.dumps(report, indent=2)
print(text)
(OUTPUT_DIR / "external-code-readiness.txt").write_text(text + "\n", encoding="utf-8")


{
  "GPECHOME": {
    "status": "ready",
    "missing_or_not_executable": []
  },
  "CHEASEHOME": {
    "status": "not configured",
    "required": false
  },
  "EFITHOME": {
    "status": "not configured",
    "required": false
  },
  "TESHOME": {
    "status": "not configured",
    "required": false
  }
}


309